# Big Five Personality Prediction on Pandora
## Towards Psychographic Audience Inference from Source Text

**Research Question:** Can LLMs reliably detect Big Five personality traits from free-form Reddit text, and what does their performance imply about the feasibility of inferring audience psychographic attributes from source text?

**Three prompt conditions per model:**
- `simple` — minimal prompt, replicates Di Cursi et al. (2025) simple condition
- `enriched` — full trait cues, replicates Di Cursi et al. (2025) complex condition  
- `audience` — novel reader-framing prompt (this paper's contribution)

**Before running:** Runtime → Change runtime type → T4 GPU → Runtime → Run All

In [ ]:
# ============================================================
# CELL 1: Install
# ============================================================
!pip install -q transformers accelerate datasets tqdm scikit-learn

In [ ]:
# ============================================================
# CELL 2: Imports
# ============================================================
import torch, gc, json, os, random
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from sklearn.metrics import classification_report, accuracy_score, f1_score
from tqdm import tqdm

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU — Runtime > Change runtime type > T4 GPU")

In [ ]:
# ============================================================
# CELL 3: Configuration
# ============================================================
SAMPLE_SIZE  = 100
RANDOM_SEED  = 42
RESULTS_DIR  = "/content/results"
TRAITS       = ["Openness", "Conscientiousness", "Extraversion", "Agreeableness", "Neuroticism"]
PROMPT_TYPES = ["simple", "enriched", "audience"]

os.makedirs(RESULTS_DIR, exist_ok=True)
random.seed(RANDOM_SEED)
print("Config ready.")

In [ ]:
# ============================================================
# CELL 4: Load dataset
# ============================================================
print("Loading dataset...")
dataset_hf = load_dataset(
    "Fatima0923/Automated-Personality-Prediction",
    download_mode="force_redownload",
    verification_mode="no_checks"
)
print("Columns:", dataset_hf["train"].column_names)

In [ ]:
# ============================================================
# CELL 5: Binarize with median-split thresholds
#
# WHY MEDIAN SPLIT:
# Pandora scores are percentiles (0-100). Using threshold=0.0
# causes near-total class imbalance — almost all users score
# above 0, so every sample becomes class 1. This makes all
# classification metrics meaningless (see initial run results).
# Median split produces ~50/50 balance consistent with the
# binary classification literature (Di Cursi et al., 2025).
# ============================================================
test_df = dataset_hf["test"].to_pandas()
thresholds = {}
trait_cols = ["openness", "conscientiousness", "extraversion", "agreeableness", "neuroticism"]

print("Thresholds (median splits):")
for col in trait_cols:
    median = test_df[col].median()
    thresholds[col.capitalize()] = median
    pct = (test_df[col] > median).mean()
    print(f"  {col}: median={median:.1f} → {pct*100:.1f}% positive")

def convert_dataset(hf_split, thresholds):
    data = []
    for row in hf_split:
        data.append({
            "comments": [row["text"]],
            "labels": {
                t: 1 if row[t.lower()] > thresholds[t] else 0
                for t in TRAITS
            }
        })
    return data

full_dataset = convert_dataset(dataset_hf["test"], thresholds)
dataset = random.sample(full_dataset, min(SAMPLE_SIZE, len(full_dataset)))
print(f"\nUsing {len(dataset)} samples")

print("\nClass balance:")
for trait in TRAITS:
    n = sum(1 for d in dataset if d["labels"][trait] == 1)
    print(f"  {trait}: {n}/{len(dataset)} positive ({n/len(dataset)*100:.1f}%)")

In [ ]:
# ============================================================
# CELL 6: Utilities
# ============================================================
def aggregate_user_comments(user_comments, max_chars=1500):
    return " ".join(user_comments)[:max_chars]

def parse_output(text):
    text = text.strip()
    for ch in text:
        if ch == "0": return 0
        if ch == "1": return 1
    return None

def save_results(results, label):
    path = os.path.join(RESULTS_DIR, f"{label}_results.json")
    with open(path, "w") as f:
        json.dump({k: {"y_true": v[0], "y_pred": v[1]} for k, v in results.items()}, f)
    print(f"  Saved → {path}")

def print_summary(results, label):
    print(f"\n{'='*55}\nSUMMARY: {label}\n{'='*55}")
    for trait in TRAITS:
        if trait not in results: continue
        y_true, y_pred = results[trait]
        if not y_true: continue
        print(f"\n--- {trait} ---")
        print(classification_report(y_true, y_pred, digits=3, zero_division=0))
        print(f"Accuracy: {accuracy_score(y_true, y_pred):.3f}  "
              f"Macro-F1: {f1_score(y_true, y_pred, average='macro', zero_division=0):.3f}")

def free_model(*names):
    """Delete model variables and flush GPU cache."""
    import __main__ as main_module
    for name in names:
        if name in globals():
            del globals()[name]
    gc.collect()
    torch.cuda.empty_cache()
    print(f"VRAM after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB")

print("Utilities ready.")

In [ ]:
# ============================================================
# CELL 7: Prompt library
#
# Sources:
#   Trait overview  — Costa & McCrae (1992) NEO-PI-R
#   Facet cues      — IPIP inventory (Goldberg et al., 2006)
#   Adjective lists — Goldberg (1990)
# ============================================================

trait_overview = {
    "Openness": (
        "High-Openness individuals are imaginative and sensitive to art and beauty. "
        "They prefer variety and are intellectually curious, creative, and open to new experiences. "
        "Low-Openness individuals are conventional, prefer routine, and favour practical thinking."
    ),
    "Conscientiousness": (
        "Conscientiousness reflects organisation, self-discipline, and goal-directed behaviour. "
        "High scorers are reliable, punctual, and thorough. "
        "Low scorers tend to be disorganised, impulsive, and careless with obligations."
    ),
    "Extraversion": (
        "Extraversion reflects sociability, assertiveness, and positive emotionality. "
        "High scorers are energetic, talkative, and seek social stimulation. "
        "Low scorers are reserved, prefer solitude, and find social interaction draining."
    ),
    "Agreeableness": (
        "Agreeableness is primarily a dimension of interpersonal behaviour. "
        "High scorers are empathetic, cooperative, and warm toward others. "
        "Low scorers are competitive, skeptical, and may be perceived as cold or antagonistic."
    ),
    "Neuroticism": (
        "Neuroticism reflects emotional instability and vulnerability to stress. "
        "High scorers experience frequent anxiety, moodiness, and emotional distress. "
        "Low scorers are calm, emotionally stable, and resilient under pressure."
    ),
}

ipip_facets = {
    "Openness": {
        "positive": [
            "Has a rich vocabulary and vivid imagination.",
            "Spends time reflecting; is full of ideas and loves challenging reading.",
            "Carries conversations to a higher intellectual level.",
            "Is quick to understand abstract concepts.",
        ],
        "negative": [
            "Has difficulty with abstract ideas; avoids challenging material.",
            "Is not interested in art or new ideas; prefers routine.",
            "Will not probe deeply into a subject.",
        ],
    },
    "Conscientiousness": {
        "positive": [
            "Is always prepared; pays close attention to details.",
            "Gets tasks done immediately; follows a schedule and sticks to plans.",
            "Continues until everything is perfect.",
        ],
        "negative": [
            "Leaves things unfinished; makes a mess of tasks.",
            "Shirks duties; finds it difficult to get down to work.",
        ],
    },
    "Extraversion": {
        "positive": [
            "Is the life of the party; feels comfortable around people.",
            "Starts conversations; does not mind being the centre of attention.",
            "Makes friends easily and is skilled in social situations.",
        ],
        "negative": [
            "Does not talk a lot; keeps in the background.",
            "Finds it difficult to approach others; is quiet around strangers.",
        ],
    },
    "Agreeableness": {
        "positive": [
            "Is interested in people; sympathises with others' feelings.",
            "Takes time for others; makes people feel at ease.",
            "Has a good word for everyone; thinks of others first.",
        ],
        "negative": [
            "Insults people; is not interested in others' problems.",
            "Feels little concern for others; is indifferent to their feelings.",
        ],
    },
    "Neuroticism": {
        "positive": [
            "Gets stressed easily; worries constantly and has frequent mood swings.",
            "Is easily disturbed; often feels blue and panics easily.",
            "Gets overwhelmed by emotions; takes offense easily.",
        ],
        "negative": [
            "Is relaxed most of the time; seldom feels blue or irritated.",
            "Is not easily bothered by things.",
        ],
    },
}

goldberg_adjectives = {
    "Openness":          {"positive": ["intelligent","philosophical","creative","curious","imaginative","insightful","intellectual"],
                          "negative":  ["simple","dull","narrow","ignorant","illogical"]},
    "Conscientiousness": {"positive": ["organised","persistent","thorough","dependable","punctual","disciplined","reliable"],
                          "negative":  ["messy","forgetful","lazy","careless","erratic","impulsive"]},
    "Extraversion":      {"positive": ["jolly","talkative","lively","outgoing","assertive","energetic","cheerful"],
                          "negative":  ["reserved","quiet","withdrawn","aloof","shy","introverted"]},
    "Agreeableness":     {"positive": ["friendly","generous","kind","warm","cooperative","patient","tactful","helpful"],
                          "negative":  ["critical","harsh","stubborn","argumentative","cold","selfish"]},
    "Neuroticism":       {"positive": ["touchy","nervous","fearful","unstable","moody","oversensitive","whiny"],
                          "negative":  ["calm","stable","confident","peaceful","resilient","unflappable"]},
}


def _facet_block(trait):
    pos = "\n".join(f"  - {f}" for f in ipip_facets[trait]["positive"])
    neg = "\n".join(f"  - {f}" for f in ipip_facets[trait]["negative"])
    pa  = ", ".join(goldberg_adjectives[trait]["positive"])
    na  = ", ".join(goldberg_adjectives[trait]["negative"])
    return pos, neg, pa, na


def build_simple_prompt(text, trait):
    """Minimal prompt — replicates Di Cursi et al. (2025) simple condition."""
    return (
        f'You are a psychological text classifier. '
        f'Decide if the Big Five trait **{trait}** is expressed in the TEXT. '
        f'Evaluate the text, not the person. '
        f'Respond ONLY with 0 or 1. Do not explain.\n\n'
        f'TEXT:\n{text}\n\nAnswer:'
    )


def build_enriched_prompt(text, trait):
    """
    Enriched prompt — replicates Di Cursi et al. (2025) complex condition.
    Combines: Costa & McCrae (1992) overview + IPIP facets + Goldberg (1990) adjectives.
    """
    pos_f, neg_f, pa, na = _facet_block(trait)
    return (
        f'You are an expert in Automatic Personality Prediction (Big Five / OCEAN).\n\n'
        f'Estimate whether **{trait}** is reflected in the style of the TEXT. '
        f'"1" = present, "0" = absent.\n'
        f'Base your judgment on tone, content, emotional framing, and vocabulary.\n\n'
        f'### Trait Overview\n{trait_overview[trait]}\n\n'
        f'### Facet-Level Cues (IPIP)\n'
        f'Signals for "1":\n{pos_f}\n\n'
        f'Signals for "0":\n{neg_f}\n\n'
        f'### Adjectives (Goldberg 1990)\n'
        f'"1" → {pa}\n'
        f'"0" → {na}\n\n'
        f'Respond ONLY with 0 or 1.\n\n'
        f'TEXT:\n{text}\n\nAnswer:'
    )


def build_audience_prompt(text, trait):
    """
    Audience inference prompt — the novel framing of this paper.

    Standard APPT (Di Cursi et al. 2025; Gjurkovic et al. 2020) asks:
      'Does the AUTHOR exhibit trait X?'
    This prompt asks:
      'Would a person HIGH in trait X be drawn to this text as a READER?'

    This operationalises psychographic audience inference:
    given a source text, predict the personality profile of its likely audience.
    """
    pos_f, neg_f, pa, na = _facet_block(trait)
    return (
        f'You are an expert in psychographic audience analysis (Big Five / OCEAN).\n\n'
        f'Do NOT assess the personality of the author.\n'
        f'Instead, determine whether a person HIGH in **{trait}** would be drawn to '
        f'READ, ENGAGE WITH, or SHARE this text.\n\n'
        f'Consider: Does the tone, topic, vocabulary, and emotional register resonate '
        f'with someone who exhibits high {trait}?\n\n'
        f'### High-{trait} reader profile\n{trait_overview[trait]}\n\n'
        f'### A high-{trait} reader tends to:\n{pos_f}\n\n'
        f'### A low-{trait} reader (less likely to engage) tends to:\n{neg_f}\n\n'
        f'### High-{trait} audience vocabulary\n'
        f'Typically: {pa}\n'
        f'Not typically: {na}\n\n'
        f'Output "1" if a high-{trait} person would engage, "0" if not.\n'
        f'Respond ONLY with 0 or 1.\n\n'
        f'TEXT:\n{text}\n\nAnswer:'
    )


def build_prompt(text, trait, prompt_type):
    """Single dispatcher used by both models."""
    if prompt_type == "simple":   return build_simple_prompt(text, trait)
    if prompt_type == "enriched": return build_enriched_prompt(text, trait)
    if prompt_type == "audience": return build_audience_prompt(text, trait)
    raise ValueError(f"Unknown prompt_type: {prompt_type}")


print("Prompt library ready (simple | enriched | audience).")

---
# Part 1: LLaMA-3-8B-Instruct

In [ ]:
# ============================================================
# CELL 8: Load LLaMA
# Verify dataset still in memory first — catches silent resets
# ============================================================
assert 'dataset' in dir() and len(dataset) == SAMPLE_SIZE, \
    "Dataset lost — re-run from Cell 1 (Runtime reset likely)"

LLAMA_NAME = "NousResearch/Meta-Llama-3-8B-Instruct"
print("Loading LLaMA tokenizer...")
llama_tokenizer = AutoTokenizer.from_pretrained(LLAMA_NAME)
llama_tokenizer.pad_token = llama_tokenizer.eos_token
llama_tokenizer.pad_token_id = llama_tokenizer.eos_token_id

print("Loading LLaMA model (3-5 min)...")
llama_model = AutoModelForCausalLM.from_pretrained(
    LLAMA_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
llama_model.eval()
print(f"LLaMA ready. VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ============================================================
# CELL 9: LLaMA inference
# FIX: input_length saved before generate() to prevent
# AttributeError on inputs["input_ids"].shape post-generation
# ============================================================
@torch.no_grad()
def predict_llama(text, trait, prompt_type="enriched"):
    prompt = build_prompt(text, trait, prompt_type)
    inputs = llama_tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=512
    ).to(llama_model.device)
    input_length = inputs["input_ids"].shape[-1]  # save BEFORE generate
    outputs = llama_model.generate(
        **inputs, max_new_tokens=5, do_sample=False,
        pad_token_id=llama_tokenizer.eos_token_id
    )
    decoded = llama_tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    return parse_output(decoded)

In [ ]:
# ============================================================
# CELL 10: LLaMA evaluation — all three prompt conditions
# Saves after every trait in case of disconnection
# ============================================================
llama_all_results = {}

for prompt_type in PROMPT_TYPES:
    print(f"\n{'='*55}")
    print(f"LLaMA | Prompt: {prompt_type.upper()}")
    print(f"{'='*55}")
    results = {}

    for trait in TRAITS:
        y_true, y_pred, n_invalid = [], [], 0
        for user in tqdm(dataset, desc=trait):
            text = aggregate_user_comments(user["comments"])
            pred = predict_llama(text, trait, prompt_type)
            if pred is None:
                n_invalid += 1
            else:
                y_true.append(user["labels"][trait])
                y_pred.append(pred)
        results[trait] = (y_true, y_pred)
        print(f"  {trait} | Invalid: {n_invalid}/{len(dataset)}")
        print(classification_report(y_true, y_pred, digits=3, zero_division=0))
        save_results(results, f"llama_{prompt_type}")

    llama_all_results[prompt_type] = results
    print_summary(results, f"LLaMA | {prompt_type}")

print("\nLLaMA evaluation complete.")

In [ ]:
# ============================================================
# CELL 11: Free LLaMA — MUST run before loading Mistral
# Both models together exceed T4 VRAM (16 GB)
# ============================================================
print(f"VRAM before cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB")
del llama_model, llama_tokenizer
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f"VRAM after cleanup:  {torch.cuda.memory_allocated()/1e9:.2f} GB")
print("LLaMA freed. Safe to load Mistral.")

---
# Part 2: Mistral-7B-Instruct

In [ ]:
# ============================================================
# CELL 12: Load Mistral
# ============================================================
MISTRAL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
print("Loading Mistral tokenizer...")
mistral_tokenizer = AutoTokenizer.from_pretrained(MISTRAL_NAME)
mistral_tokenizer.pad_token = mistral_tokenizer.eos_token
mistral_tokenizer.pad_token_id = mistral_tokenizer.eos_token_id

print("Loading Mistral model (3-5 min)...")
mistral_model = AutoModelForCausalLM.from_pretrained(
    MISTRAL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
mistral_model.eval()
print(f"Mistral ready. VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ============================================================
# CELL 13: Mistral inference
# Uses chat-template format required by Mistral instruct models
# ============================================================
@torch.no_grad()
def predict_mistral(text, trait, prompt_type="enriched"):
    prompt = build_prompt(text, trait, prompt_type)
    messages = [{"role": "user", "content": prompt}]
    encoded = mistral_tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True,
        truncation=True, max_length=512
    ).to(mistral_model.device)
    input_length = encoded.shape[-1]  # save BEFORE generate
    outputs = mistral_model.generate(
        encoded, max_new_tokens=5, do_sample=False,
        temperature=None, top_p=None,
        pad_token_id=mistral_tokenizer.eos_token_id
    )
    decoded = mistral_tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    return parse_output(decoded)

In [ ]:
# ============================================================
# CELL 14: Mistral evaluation — all three prompt conditions
# ============================================================
mistral_all_results = {}

for prompt_type in PROMPT_TYPES:
    print(f"\n{'='*55}")
    print(f"Mistral | Prompt: {prompt_type.upper()}")
    print(f"{'='*55}")
    results = {}

    for trait in TRAITS:
        y_true, y_pred, n_invalid = [], [], 0
        for user in tqdm(dataset, desc=trait):
            text = aggregate_user_comments(user["comments"])
            pred = predict_mistral(text, trait, prompt_type)
            if pred is None:
                n_invalid += 1
            else:
                y_true.append(user["labels"][trait])
                y_pred.append(pred)
        results[trait] = (y_true, y_pred)
        print(f"  {trait} | Invalid: {n_invalid}/{len(dataset)}")
        print(classification_report(y_true, y_pred, digits=3, zero_division=0))
        save_results(results, f"mistral_{prompt_type}")

    mistral_all_results[prompt_type] = results
    print_summary(results, f"Mistral | {prompt_type}")

print("\nMistral evaluation complete.")

In [ ]:
# ============================================================
# CELL 15: Free Mistral before audience extension
# ============================================================
print(f"VRAM before cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB")
del mistral_model, mistral_tokenizer
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f"VRAM after cleanup:  {torch.cuda.memory_allocated()/1e9:.2f} GB")
print("Mistral freed. Ready for audience extension.")

---
# Part 3: Audience Inference Extension

**Design:**  
Standard APPT asks: *given text person X wrote, what are their traits?*  
This extension asks: *given a source text, what personality profile would its likely audience have?*

**Method:**  
1. Select users with extreme scores (top/bottom 15%) on a focal trait from the training split  
2. Aggregate their Reddit posts into a single source document  
3. Ask the model (audience prompt): would a high-X person engage with this?  
4. Compare prediction against the known ground truth profile of that group  

If predictions align → early feasibility evidence  
If they do not → demonstrates the author→audience transfer gap  
Either outcome is a valid finding.

In [ ]:
# ============================================================
# CELL 16: Reload Mistral for audience extension
# (using Mistral as it had lower invalid rate in main eval)
# ============================================================
MISTRAL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
print("Reloading Mistral for audience extension...")
mistral_tokenizer = AutoTokenizer.from_pretrained(MISTRAL_NAME)
mistral_tokenizer.pad_token = mistral_tokenizer.eos_token
mistral_tokenizer.pad_token_id = mistral_tokenizer.eos_token_id
mistral_model = AutoModelForCausalLM.from_pretrained(
    MISTRAL_NAME, torch_dtype=torch.float16, device_map="auto"
)
mistral_model.eval()
print(f"Mistral ready. VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ============================================================
# CELL 17: Audience inference experiment
# ============================================================
train_df = dataset_hf["train"].to_pandas()

def get_extreme_group(df, trait, direction="high", pct=0.15, max_chars=1500):
    col = trait.lower()
    q = df[col].quantile(1 - pct if direction == "high" else pct)
    group = df[df[col] >= q] if direction == "high" else df[df[col] <= q]
    source_text = " ".join(group["text"].tolist())[:max_chars]
    means = {t: group[t].mean() for t in trait_cols}
    return source_text, means, len(group)

test_cases = [
    ("Openness",          "high", "High-Openness audience (curious, imaginative)"),
    ("Openness",          "low",  "Low-Openness audience (conventional, practical)"),
    ("Neuroticism",       "high", "High-Neuroticism audience (anxious, emotionally sensitive)"),
    ("Neuroticism",       "low",  "Low-Neuroticism audience (calm, stable)"),
    ("Conscientiousness", "high", "High-Conscientiousness audience (organised, disciplined)"),
    ("Agreeableness",     "high", "High-Agreeableness audience (empathetic, cooperative)"),
]

extension_results = []

print("="*55)
print("AUDIENCE INFERENCE EXTENSION")
print("Model: Mistral | Prompt: audience")
print("="*55)

for focal_trait, direction, label in test_cases:
    print(f"\n--- {label} ---")
    source_text, means, n = get_extreme_group(train_df, focal_trait, direction)
    print(f"Group size: {n} users")
    print("Ground truth mean scores:")
    for t, score in means.items():
        print(f"  {t.capitalize():<22} {score:.1f}")

    print("\nModel predicted audience profile:")
    predicted = {}
    for trait in TRAITS:
        pred = predict_mistral(source_text, trait, prompt_type="audience")
        predicted[trait] = pred
        gt    = means[trait.lower()]
        gt_lbl   = "HIGH" if gt > 50 else "LOW "
        pred_lbl = "HIGH" if pred == 1 else "LOW " if pred == 0 else "INV "
        match = "✓" if (pred == 1 and gt > 50) or (pred == 0 and gt <= 50) else "✗"
        print(f"  {trait:<22} pred={pred_lbl}  gt={gt_lbl}({gt:.0f})  {match}")

    extension_results.append({
        "label": label, "focal_trait": focal_trait, "direction": direction,
        "n_users": n, "ground_truth_means": means, "predicted_profile": predicted
    })

with open(os.path.join(RESULTS_DIR, "extension_results.json"), "w") as f:
    json.dump(extension_results, f, indent=2)
print("\nExtension results saved.")

---
# Part 4: Final Comparison Table

In [ ]:
# ============================================================
# CELL 18: Full results table
# Loads from saved JSON so this works even after a disconnection
# ============================================================
def load_results(label):
    path = os.path.join(RESULTS_DIR, f"{label}_results.json")
    if not os.path.exists(path):
        return None
    with open(path) as f:
        raw = json.load(f)
    return {k: (v["y_true"], v["y_pred"]) for k, v in raw.items()}

print("\n" + "="*80)
print("FULL RESULTS — Macro-F1 by Model × Prompt Type × Trait")
print("Benchmark: Di Cursi et al. (2025) Pandora ~0.50-0.55 macro-F1")
print("="*80)

col_order = [("llama","simple"),("llama","enriched"),("llama","audience"),
             ("mistral","simple"),("mistral","enriched"),("mistral","audience")]

header = f"{'Trait':<22}" + "".join(f"  {m[:3]}_{p[:3]}" for m,p in col_order)
print(header)
print("-"*80)

for trait in TRAITS:
    row = f"{trait:<22}"
    for model, pt in col_order:
        res = load_results(f"{model}_{pt}")
        if res and trait in res:
            yt, yp = res[trait]
            f1 = f1_score(yt, yp, average='macro', zero_division=0)
            row += f"  {f1:.3f}"
        else:
            row += "      —"
    print(row)

print("\nNotes:")
print("  simple/enriched  — replicate Di Cursi et al. (2025) conditions")
print("  audience         — novel reader-framing prompt (this paper)")
print(f"\nAll results saved to: {RESULTS_DIR}")